# Checkpoint 5 — Análise Estatística e Clusterização de Vinhos

**Disciplinas:** Statistical Computing with R & Python; Machine Learning & Modelling  
**Integrantes:** preencher nome completo e matrícula de cada integrante antes da entrega.

## Objetivo
Na Parte 1, vamos explorar duas variáveis quantitativas do conjunto de vinhos, calculando frequências, medidas descritivas, quartis e probabilidades empíricas. Na Parte 2, vamos padronizar as variáveis numéricas e aplicar K-means para investigar grupos de vinhos semelhantes.

> Execute as células em ordem. Confira os resultados gerados no seu ambiente e adapte as interpretações finais aos números que aparecerem. O notebook tenta ler `wines.csv` na mesma pasta; se o arquivo não estiver lá, usa o conjunto Wine incluído no scikit-learn, que contém as mesmas 13 características químicas e uma coluna de classe.


In [ ]:
# Bibliotecas usadas no projeto.
# Se alguma biblioteca não estiver instalada, execute no terminal:
# pip install pandas numpy matplotlib seaborn scikit-learn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Carregamento e conhecimento dos dados

Primeiro, tentamos abrir `wines.csv` localmente. Se o arquivo não estiver disponível, carregamos a versão Wine que acompanha o scikit-learn. A coluna de classe, quando existir, não será usada no agrupamento, porque o objetivo é formar clusters com base nas características químicas, sem fornecer a classe ao algoritmo.

In [ ]:
arquivo = "wine.csv" if os.path.exists("wine.csv") else "wines.csv"

if os.path.exists(arquivo):
    df = pd.read_csv(arquivo)
    print("Arquivo wines.csv carregado.")
else:
    wine = load_wine(as_frame=True)
    df = wine.frame.copy()
    # Renomeia a variável-alvo para deixar claro que é uma classe conhecida.
    df = df.rename(columns={"target": "class"})
    print("Arquivo CSV não encontrado; usando o dataset Wine do scikit-learn.")

display(df.head())
print("Dimensões:", df.shape)
print("\nTipos de dados:")
display(df.dtypes.to_frame("tipo"))
print("\nValores ausentes por coluna:")
display(df.isna().sum().to_frame("valores_ausentes"))


## 2. Preparação inicial

Vamos selecionar duas variáveis quantitativas para a análise estatística. Usaremos **alcohol** (teor alcoólico) e **malic_acid** (ácido málico), se essas colunas existirem. Como os arquivos podem ter nomes diferentes, o código procura nomes equivalentes sem diferenciar maiúsculas de minúsculas. Se necessário, confira `df.columns` e ajuste os nomes.

In [ ]:
print("Colunas disponíveis:", list(df.columns))

# Padroniza apenas os nomes para facilitar a identificação das colunas.
def normalizar_nome(nome):
    return str(nome).strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_").replace("/", "_")

mapa_colunas = {normalizar_nome(c): c for c in df.columns}

def encontrar_coluna(opcoes):
    for opcao in opcoes:
        if opcao in mapa_colunas:
            return mapa_colunas[opcao]
    return None

col_alcohol = encontrar_coluna(["alcohol", "alco", "teor_alcoolico", "teor_de_alcool"])
col_malic = encontrar_coluna(["malic_acid", "malicacid", "acido_malico", "malic_acid_g_l"])

if col_alcohol is None or col_malic is None:
    raise ValueError(
        "Não encontrei as colunas de teor alcoólico e ácido málico. "
        "Confira os nomes impressos acima e ajuste col_alcohol e col_malic."
    )

# Converte para numérico e remove apenas linhas sem valores nessas duas variáveis.
dados_estat = df[[col_alcohol, col_malic]].copy()
dados_estat.columns = ["alcohol", "malic_acid"]
for coluna in dados_estat.columns:
    dados_estat[coluna] = pd.to_numeric(dados_estat[coluna], errors="coerce")
dados_estat = dados_estat.dropna()

print("Registros válidos para a análise estatística:", len(dados_estat))
display(dados_estat.describe())


## 3. Parte 1(a) — Tabela de distribuição de frequências e gráficos

Para facilitar a leitura, agrupamos cada variável em intervalos. A tabela apresenta frequência absoluta, frequência relativa e frequência acumulada. O histograma mostra a forma da distribuição e ajuda a observar concentração, dispersão e possíveis assimetrias.

In [ ]:
def tabela_frequencias(serie, numero_classes=6):
    # O pandas cria intervalos de mesma amplitude; include_lowest inclui o menor valor.
    categorias = pd.cut(serie, bins=numero_classes, include_lowest=True)
    freq_abs = categorias.value_counts(sort=False)
    tabela = pd.DataFrame({
        "frequencia_absoluta": freq_abs,
        "frequencia_relativa": freq_abs / len(serie),
        "frequencia_acumulada": freq_abs.cumsum()
    })
    tabela["frequencia_relativa_%"] = tabela["frequencia_relativa"] * 100
    return tabela

for coluna in ["alcohol", "malic_acid"]:
    print(f"\nTabela de frequências — {coluna}")
    display(tabela_frequencias(dados_estat[coluna], numero_classes=6).round(3))

    plt.figure(figsize=(8, 4))
    sns.histplot(data=dados_estat, x=coluna, bins=6, edgecolor="black")
    plt.title(f"Distribuição de frequência: {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()

# Interpretação: compare onde se concentram as observações, se a distribuição é simétrica
# ou assimétrica e se há intervalos com frequência muito maior que os demais.


## 4. Parte 1(b) — Análise descritiva

Calculamos medidas de tendência central (média, mediana e moda), dispersão (amplitude, variância, desvio-padrão e intervalo interquartil) e separatrizes (quartis). A variância e o desvio-padrão abaixo são amostrais, usando `ddof=1`.

In [ ]:
def resumo_descritivo(serie):
    moda = serie.mode()
    return pd.Series({
        "média": serie.mean(),
        "mediana": serie.median(),
        "moda (primeira, se houver empate)": moda.iloc[0] if len(moda) else np.nan,
        "mínimo": serie.min(),
        "máximo": serie.max(),
        "amplitude": serie.max() - serie.min(),
        "variância amostral": serie.var(ddof=1),
        "desvio-padrão amostral": serie.std(ddof=1),
        "Q1 (25%)": serie.quantile(0.25),
        "Q2 (50%)": serie.quantile(0.50),
        "Q3 (75%)": serie.quantile(0.75),
        "intervalo interquartil": serie.quantile(0.75) - serie.quantile(0.25)
    })

for coluna in ["alcohol", "malic_acid"]:
    print(f"\nMedidas descritivas — {coluna}")
    display(resumo_descritivo(dados_estat[coluna]).to_frame("valor").round(4))

# Interpretação: média e mediana próximas sugerem menor assimetria, embora não a provem.
# O desvio-padrão mede a dispersão em torno da média; o IQR descreve os 50% centrais.
# Q1 marca aproximadamente 25% dos valores abaixo dele e Q3, 75%.


## 5. Parte 1(c) — Distribuição de probabilidade e cálculos probabilísticos

Como as variáveis são quantitativas contínuas, usaremos probabilidades empíricas: a proporção de observações que atende a uma condição. Isso permite responder perguntas como “qual a probabilidade de um vinho ter teor alcoólico acima de 13%?” e “qual a probabilidade de o ácido málico ficar abaixo de determinado valor?”. Os limites são escolhidos a partir da mediana e do terceiro quartil do próprio conjunto.

In [ ]:
# Probabilidade empírica: número de casos favoráveis dividido pelo total de casos válidos.
def probabilidade_empirica(condicao):
    return condicao.mean()

# Cálculo 1: teor alcoólico acima de 13%.
p_alcohol_13 = probabilidade_empirica(dados_estat["alcohol"] > 13)

# Cálculo 2: ácido málico abaixo ou igual ao terceiro quartil.
q3_malic = dados_estat["malic_acid"].quantile(0.75)
p_malic_q3 = probabilidade_empirica(dados_estat["malic_acid"] <= q3_malic)

# Cálculo 3 adicional: probabilidade de um registro estar acima da mediana de teor alcoólico.
mediana_alcohol = dados_estat["alcohol"].median()
p_alcohol_mediana = probabilidade_empirica(dados_estat["alcohol"] > mediana_alcohol)

print(f"P(álcool > 13) = {p_alcohol_13:.3f} ({p_alcohol_13*100:.1f}%)")
print(f"P(ácido málico <= Q3={q3_malic:.3f}) = {p_malic_q3:.3f} ({p_malic_q3*100:.1f}%)")
print(f"P(álcool > mediana={mediana_alcohol:.3f}) = {p_alcohol_mediana:.3f} ({p_alcohol_mediana*100:.1f}%)")

# Interpretação: cada valor representa a proporção observada no dataset, não uma garantia
# sobre todos os vinhos existentes. Em um conjunto de dados finito, os resultados são estimativas.


# Parte 2 — Machine Learning & Modelling: K-means

## 6. Introdução

O objetivo é investigar se os vinhos podem ser separados em grupos de características químicas semelhantes. O K-means divide as observações em K grupos, procurando aproximar cada observação do centro do grupo atribuído. Como as variáveis têm escalas diferentes, é necessário padronizá-las antes do algoritmo; caso contrário, variáveis com números maiores poderiam dominar as distâncias.

## 7. Pré-processamento

Removemos uma eventual coluna de classe/target, porque ela representa uma categoria conhecida e não deve ser usada como entrada do agrupamento não supervisionado. Mantemos as variáveis numéricas, tratamos valores ausentes pela mediana e padronizamos as features com StandardScaler.

In [ ]:
# Identifica possíveis colunas de classe/identificação para não usá-las no K-means.
colunas_excluir = [
    c for c in df.columns
    if normalizar_nome(c) in {"class", "classe", "target", "label", "wine", "customer_id", "id"}
]
features = df.drop(columns=colunas_excluir, errors="ignore").select_dtypes(include=np.number).copy()

# Exclui colunas numéricas totalmente vazias e remove features constantes.
features = features.dropna(axis=1, how="all")
features = features.loc[:, features.nunique(dropna=True) > 1]

# Imputação simples: valores ausentes são substituídos pela mediana da própria coluna.
features = features.fillna(features.median(numeric_only=True))

if features.shape[1] < 2:
    raise ValueError("São necessárias pelo menos duas features numéricas para aplicar K-means.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print("Número de observações:", features.shape[0])
print("Número de features usadas:", features.shape[1])
print("Features:", list(features.columns))
print("Valores ausentes após tratamento:", int(features.isna().sum().sum()))


## 8. Escolha do número de grupos: Elbow e Silhouette Score

O método Elbow compara a soma das distâncias quadráticas dentro dos clusters (inertia). Procuramos um ponto em que a redução da inertia passe a ser menos acentuada. O Silhouette Score varia de -1 a 1; valores mais altos costumam indicar grupos mais bem separados. Nenhum dos métodos deve ser interpretado isoladamente.

In [ ]:
resultados = []
n = len(features)
k_max = min(10, n - 1)

for k in range(2, k_max + 1):
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    rotulos = modelo.fit_predict(X_scaled)
    resultados.append({
        "k": k,
        "inertia": modelo.inertia_,
        "silhouette": silhouette_score(X_scaled, rotulos)
    })

metricas = pd.DataFrame(resultados)
display(metricas.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(metricas["k"], metricas["inertia"], marker="o")
ax.set_title("Método Elbow")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Inertia")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(metricas["k"], metricas["silhouette"], marker="o")
ax.set_title("Silhouette Score por número de clusters")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

k_silhouette = int(metricas.loc[metricas["silhouette"].idxmax(), "k"])
print("Melhor k segundo o Silhouette Score:", k_silhouette)
print("Use o gráfico Elbow como segunda evidência e justifique a escolha final.")


## 9. Ajuste final do K-means e análise dos grupos

A célula seguinte usa o melhor K segundo o Silhouette Score como ponto de partida. Confira também o gráfico Elbow: se ele indicar claramente outro K próximo, ajuste `k_final` e explique a decisão no relatório. Para interpretar os grupos, calculamos médias das variáveis originais por cluster, permitindo comparar teor alcoólico, ácido málico, cinzas e demais características.

In [ ]:
# Escolha inicial automática pelo maior Silhouette Score.
# Se o método Elbow apontar claramente outro valor, altere k_final e justifique a escolha.
k_final = k_silhouette

modelo_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df_resultado = df.loc[features.index].copy()
df_resultado["cluster"] = modelo_final.fit_predict(X_scaled)

print("Quantidade de vinhos em cada grupo:")
display(df_resultado["cluster"].value_counts().sort_index().to_frame("quantidade"))

# Estatísticas das features originais para descrever o perfil de cada grupo.
perfil_clusters = df_resultado.groupby("cluster")[list(features.columns)].mean().round(3)
print("Média das características por cluster:")
display(perfil_clusters)

# Compara variáveis centrais para a pergunta do enunciado.
colunas_interesse = [c for c in features.columns if normalizar_nome(c) in
                     {"alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium",
                      "total_phenols", "flavanoids", "color_intensity", "hue", "proline"}]
if colunas_interesse:
    display(df_resultado.groupby("cluster")[colunas_interesse].mean().round(3))

# Em geral, o cluster com maior média de alcohol tem maior teor alcoólico médio.
# O cluster com maior média de malic_acid tem maior ácido málico médio.
# Leia a tabela para identificar os grupos e escreva a interpretação com os valores obtidos.


In [ ]:
# Identificação automática dos grupos extremos para facilitar a interpretação.
if col_alcohol in df_resultado.columns:
    media_alcool = df_resultado.groupby("cluster")[col_alcohol].mean()
    print(f"Cluster com maior teor alcoólico médio: {media_alcool.idxmax()} "
          f"(média = {media_alcool.max():.3f})")
if col_malic in df_resultado.columns:
    media_acido = df_resultado.groupby("cluster")[col_malic].mean()
    print(f"Cluster com maior ácido málico médio: {media_acido.idxmax()} "
          f"(média = {media_acido.max():.3f})")

# Visualização de dois atributos: cada ponto é um vinho e a cor representa o cluster.
if col_alcohol in df_resultado.columns and col_malic in df_resultado.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df_resultado, x=col_alcohol, y=col_malic,
                    hue=df_resultado["cluster"].astype(str), palette="deep", s=65)
    plt.title("Clusters de vinhos: teor alcoólico e ácido málico")
    plt.xlabel("Teor alcoólico")
    plt.ylabel("Ácido málico")
    plt.tight_layout()
    plt.show()


## 10. Conclusão — preencher após executar

Com base nos resultados obtidos, conclua:
1. Quais foram as principais características das distribuições de teor alcoólico e ácido málico?
2. O que média, mediana, desvio-padrão e quartis revelaram?
3. Quais probabilidades empíricas foram calculadas e como devem ser interpretadas?
4. Qual valor de K foi escolhido e como Elbow e Silhouette apoiaram a decisão?
5. Qual cluster apresentou maior teor alcoólico médio? Qual apresentou maior ácido málico médio? Cite os valores da tabela.
6. Quais são as limitações: o K-means depende da escala, da escolha de K e da forma dos dados; clusters são padrões exploratórios e não significam necessariamente categorias oficiais de vinho.

**Antes de entregar:** execute todas as células, confira se os nomes das colunas foram reconhecidos, complete a conclusão com os valores reais gerados, inclua nome e matrícula de todos os integrantes e disponibilize o notebook no GitHub.
